# Trace Count v8: needle-count stress test

目标：把 v2-style setting 拉长、needle 数量增多，找出 accuracy 开始下降的 count 区间。

输出重点：

1. autoregressive final-count accuracy by gold count；
2. `tf_accuracy` / `ar_accuracy` 同时保存，图和阈值默认使用 `ar_accuracy`；
3. first count below 0.9 accuracy；
4. CoT 和 non-thinking 的崩塌阈值是否不同。
            

## 1. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
INSTALL_DEPS = False

if IN_COLAB:
    repo_dir = Path("/content/Synthetic_CoT_NiaH_Count")
    cwd = Path.cwd()
    if (cwd / ".git").exists() or (cwd / "README.md").exists():
        repo_dir = cwd
    elif not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "pandas", "matplotlib", "tqdm"], check=True)

import pandas as pd
from IPython.display import Markdown, display, Image

display(Markdown(f"**Repo root:** `{ROOT}`"))

## 2. Runtime settings

In [ ]:
PRESET = "debug"  # "debug" or "main"
OUT_ROOT = "runs/trace_count_v8_many_needles"
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
SKIP_COMPLETED = True
print({"PRESET": PRESET, "OUT_ROOT": OUT_ROOT, "DEVICE": DEVICE})
            

## 3. Run v8 sweep

In [ ]:
from synthetic_counting_extensions.v7_v8_sweeps import run_sweep

combined = run_sweep("v8", PRESET, OUT_ROOT, skip_completed=SKIP_COMPLETED, device=DEVICE)
display(combined.head())
threshold = []
for (setting, mode), g in combined.groupby(["setting", "mode"]):
    bad = g[g["accuracy"] < 0.9]
    threshold.append({"setting": setting, "mode": mode, "first_count_below_0.9": int(bad["count"].iloc[0]) if len(bad) else "none"})
display(pd.DataFrame(threshold))
            

## 4. Display reports

In [ ]:
for run in Path(OUT_ROOT).glob("v8_*"):
    report = run / "report" / "report.html"
    fig = run / "figures" / "accuracy_by_count.png"
    if fig.exists():
        display(Markdown(f"## {run.name}\nReport: `{report}`"))
        display(Image(filename=str(fig)))
            

## 5. Save / GitHub / disconnect

In [ ]:
# Save result folder to Google Drive.
SAVE_TO_DRIVE = True
DRIVE_DEST_ROOT = "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results"

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_root = Path(DRIVE_DEST_ROOT)
    dest_root.mkdir(parents=True, exist_ok=True)
    if "RUN_DIR" in globals() and RUN_DIR is not None:
        src = Path(RUN_DIR)
    elif "OUT_ROOT" in globals():
        src = Path(OUT_ROOT)
    else:
        src = None
    if src is not None and src.exists():
        dest = dest_root / src.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        display(Markdown(f"**Saved to Drive:** `{dest}`"))
    else:
        display(Markdown("No RUN_DIR/OUT_ROOT found to save."))
else:
    display(Markdown("Drive save skipped."))

In [ ]:
# Optional: commit and push notebook/code changes to GitHub.
PUSH_TO_GITHUB = False
GIT_COMMIT_MESSAGE = "Add synthetic counting experiment notebook"

if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=False)
    subprocess.run(["git", "add", "notebooks", "synthetic_counting_extensions", "scripts"], check=True)
    subprocess.run(["git", "commit", "-m", GIT_COMMIT_MESSAGE], check=True)
    subprocess.run(["git", "push"], check=True)
else:
    display(Markdown("GitHub push skipped. Set `PUSH_TO_GITHUB = True` after checking the diff."))

In [ ]:
# Optional: disconnect Colab runtime after saving.
AUTO_DISCONNECT_AFTER_SAVE = False

if AUTO_DISCONNECT_AFTER_SAVE and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()
else:
    display(Markdown("Auto-disconnect skipped."))